# Detector de huevos v2: quitar el atajo del fondo, entrenar, exportar a TFLite

**Por qué hay un v2.** En `v1` todas las fotos de Intact son del montaje (fondo gris, base negra) y todas las fotos de otras fuentes son Crack. El modelo puede acertar en test con la regla "fondo gris = sano, otro fondo = rajado", pero en video en vivo un huevo sano sobre una mesa saldría Crack.

**Qué hace este notebook** (ejecutar de arriba abajo, en Colab con GPU):
1. Setup: GPU, `ultralytics`, Drive y dataset en `/content/eggs_v2`.
2. Genera imágenes sintéticas intercambiando huevos y fondos (solo dentro de cada split, sin fugas entre train/valid/test).
3. Mide cuánto falla `v1` en esas imágenes (diagnóstico del atajo).
4. Entrena `v2` partiendo de los pesos de `v1`.
5. Compara `v1` y `v2`, elige el mejor y calcula el umbral.
6. Exporta a `.tflite` con entrada NHWC (lo que entrega la cámara) en FP32 e INT8 y verifica que dan lo mismo que el `.pt`.

Nada de lo anterior se sobrescribe: el run nuevo se llama `v2` y el export va a `MyDrive/eggs_v2/exports/<run>`.

## 1. Setup
Comprueba la GPU, instala `ultralytics`, monta Drive y descomprime el dataset en `/content/eggs_v2` (solo si no está ya). Define las rutas que usa el resto del notebook.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics

import zipfile
from pathlib import Path
import yaml
from google.colab import drive
import ultralytics

drive.mount('/content/drive')
ultralytics.checks()

DRIVE_DIR = Path('/content/drive/MyDrive/eggs_v2')
DATA_DIR = Path('/content/eggs_v2')          # dataset original
V2_DIR = Path('/content/eggs_v2s')           # dataset original + sintéticas
RUNS_DIR = DRIVE_DIR / 'runs'
EXPORTS_DIR = DRIVE_DIR / 'exports'
BASE_RUN, NEW_RUN = 'v1', 'v2'
CLASSES = ['Crack', 'Intact']
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

if not (DATA_DIR / 'train').exists():
    for n in ['eggs_v2_parte1_train.zip', 'eggs_v2_parte2_train.zip',
              'eggs_v2_parte3_train.zip', 'eggs_v2_parte4_valid_test.zip']:
        with zipfile.ZipFile(DRIVE_DIR / n) as zf:
            zf.extractall(DATA_DIR)
        print('OK', n)
with open(DATA_DIR / 'data.yaml', 'w') as f:
    yaml.safe_dump({'path': str(DATA_DIR), 'train': str(DATA_DIR / 'train/images'),
                    'val': str(DATA_DIR / 'valid/images'), 'test': str(DATA_DIR / 'test/images'),
                    'nc': 2, 'names': CLASSES}, f, sort_keys=False)
assert (RUNS_DIR / BASE_RUN / 'weights/best.pt').exists(), 'Falta el best.pt de v1 en Drive'

import torch
libre, total = torch.cuda.mem_get_info()
print(f'GPU libre: {libre / 1e9:.1f} de {total / 1e9:.1f} GB')
if libre < 0.7 * total:
    print('AVISO: otro proceso está usando la GPU (¿otro notebook conectado al mismo Colab?). '
          'Ciérralo o reinicia su kernel antes de seguir:')
    !nvidia-smi --query-compute-apps=pid,used_memory --format=csv
print('Dataset listo en', DATA_DIR)

## 2. Funciones para generar imágenes sintéticas
Recorta cada huevo con una máscara elíptica suavizada y lo pega sobre otro fondo. Tres tipos:
- **A**: huevo Intact (del montaje) tapando un huevo Crack de otra fuente → Intact fuera del montaje.
- **B**: huevo Crack de otra fuente tapando el huevo del montaje → Crack dentro del montaje.
- **P**: 1–2 huevos de ambas clases sobre fondos procedurales (color, degradado, textura).

A veces blanquea el huevo (huevos blancos sanos), dibuja la mira verde en ambas clases y baja la resolución de toda la imagen, para que ni el color, ni la mira, ni la nitidez delaten la clase. Esta celda solo define funciones.

In [ ]:
import random
import shutil
from collections import Counter

import cv2
import numpy as np


def fuente(stem):
    """'montaje' = fotos del montaje de fondo gris (todas las Intact y parte de las Crack)."""
    return 'montaje' if stem.startswith('ec_egg') else 'otras'


def cargar_split(split_dir):
    """Lista de dicts {img, cls, box (x1,y1,x2,y2 en px), fuente} de las imágenes originales (sin _dup)."""
    items = []
    for img in sorted((split_dir / 'images').iterdir()):
        if img.suffix.lower() not in IMG_EXTS or '_dup' in img.stem:
            continue
        lines = (split_dir / 'labels' / f'{img.stem}.txt').read_text().split('\n')
        parts = lines[0].split()
        if len(parts) != 5:
            continue
        h, w = cv2.imread(str(img)).shape[:2]
        c, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
        box = ((xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h)
        items.append({'img': img, 'cls': c, 'box': box, 'fuente': fuente(img.stem), 'size': (w, h)})
    return items


def recortar_huevo(item):
    """Recorta el huevo con una máscara elíptica suavizada (los huevos son casi elipses)."""
    img = cv2.imread(str(item['img']))
    x1, y1, x2, y2 = [int(round(v)) for v in item['box']]
    x1, y1 = max(x1, 0), max(y1, 0)
    x2, y2 = min(x2, img.shape[1]), min(y2, img.shape[0])
    crop = img[y1:y2, x1:x2].copy()
    h, w = crop.shape[:2]
    mask = np.zeros((h, w), np.float32)
    cv2.ellipse(mask, (w // 2, h // 2), (max(w // 2 - 1, 1), max(h // 2 - 1, 1)), 0, 0, 360, 1.0, -1)
    k = max(3, (min(w, h) // 25) | 1)
    mask = cv2.GaussianBlur(mask, (k, k), 0)
    return crop, mask


def variar_huevo(crop, rng):
    """Volteo aleatorio y, a veces, 'blanquear' el huevo (hay huevos blancos sanos en la vida real)."""
    if rng.random() < 0.5:
        crop = crop[:, ::-1]
    if rng.random() < 0.3:
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[..., 1] *= rng.uniform(0.1, 0.4)
        hsv[..., 2] = np.clip(hsv[..., 2] * rng.uniform(1.1, 1.4) + 20, 0, 255)
        crop = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    return np.ascontiguousarray(crop)


def pegar(bg, crop, mask, cx, cy, w, h):
    """Pega el huevo (w x h px) centrado en (cx, cy). Devuelve la caja visible o None si queda muy cortado."""
    w, h = max(int(w), 4), max(int(h), 4)
    crop = cv2.resize(crop, (w, h), interpolation=cv2.INTER_AREA if w < crop.shape[1] else cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (w, h))[..., None]
    H, W = bg.shape[:2]
    x1, y1 = int(cx - w / 2), int(cy - h / 2)
    bx1, by1, bx2, by2 = max(x1, 0), max(y1, 0), min(x1 + w, W), min(y1 + h, H)
    if (bx2 - bx1) * (by2 - by1) < 0.85 * w * h:
        return None
    roi = bg[by1:by2, bx1:bx2].astype(np.float32)
    c = crop[by1 - y1:by2 - y1, bx1 - x1:bx2 - x1].astype(np.float32)
    m = mask[by1 - y1:by2 - y1, bx1 - x1:bx2 - x1]
    bg[by1:by2, bx1:bx2] = (c * m + roi * (1 - m)).astype(np.uint8)
    return bx1, by1, bx2, by2


def fondo_procedural(rng, W, H):
    """Fondos variados (color liso con ruido, degradado o textura) que no pertenecen a ninguna clase."""
    tipo = rng.randrange(3)
    c1 = np.array([rng.randrange(256) for _ in range(3)], np.float32)
    if tipo == 0:
        bg = np.ones((H, W, 3), np.float32) * c1
    elif tipo == 1:
        c2 = np.array([rng.randrange(256) for _ in range(3)], np.float32)
        t = np.linspace(0, 1, W if rng.random() < 0.5 else H, dtype=np.float32)
        t = t[None, :, None] if len(t) == W else t[:, None, None]
        bg = np.broadcast_to(c1 * (1 - t) + c2 * t, (H, W, 3)).copy()
    else:
        ruido = np.random.default_rng(rng.randrange(1 << 30)).random((H // 16 + 1, W // 16 + 1, 1)).astype(np.float32)
        ruido = cv2.resize(ruido, (W, H), interpolation=cv2.INTER_CUBIC)[..., None]
        bg = c1 * (0.6 + 0.8 * ruido)
    bg += np.random.default_rng(rng.randrange(1 << 30)).normal(0, rng.uniform(2, 12), (H, W, 3)).astype(np.float32)
    return np.clip(bg, 0, 255).astype(np.uint8)


def mira_verde(img, rng):
    """La mira verde del montaje, dibujada en imágenes de ambas clases para que no sea pista de Intact."""
    H, W = img.shape[:2]
    cx, cy = int(W * rng.uniform(0.4, 0.6)), int(H * rng.uniform(0.4, 0.6))
    color = (int(rng.uniform(80, 140)), int(rng.uniform(170, 230)), int(rng.uniform(80, 140)))
    cv2.line(img, (0, cy), (W, cy), color, 1)
    cv2.line(img, (cx, 0), (cx, H), color, 1)


def retocar(img, cajas, rng):
    """Mira verde en ambas clases y, a veces, bajar la resolución de toda la imagen para que la nitidez
    tampoco delate la clase (los huevos Intact vienen de fotos de 224 px)."""
    if rng.random() < 0.4:
        mira_verde(img, rng)
    if rng.random() < 0.5:
        H, W = img.shape[:2]
        f = rng.uniform(224, 320) / max(H, W)
        if f < 1:
            img = cv2.resize(img, (max(int(W * f), 1), max(int(H * f), 1)), interpolation=cv2.INTER_AREA)
            cajas = [(c, x1 * f, y1 * f, x2 * f, y2 * f) for c, x1, y1, x2, y2 in cajas]
    return img, cajas


def guardar(out_dir, nombre, img, cajas):
    """cajas: lista de (cls, x1, y1, x2, y2) en px -> .jpg + .txt YOLO."""
    H, W = img.shape[:2]
    cv2.imwrite(str(out_dir / 'images' / f'{nombre}.jpg'), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
    lineas = [f'{c} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}'
              for c, x1, y1, x2, y2 in cajas]
    (out_dir / 'labels' / f'{nombre}.txt').write_text('\n'.join(lineas) + '\n')


def intercambio(fondo_item, huevo_item, rng):
    """Tapa el huevo del fondo con el otro huevo (un 12-25 % más grande para cubrirlo entero)."""
    bg = cv2.imread(str(fondo_item['img']))
    crop, mask = recortar_huevo(huevo_item)
    crop = variar_huevo(crop, rng)
    x1, y1, x2, y2 = fondo_item['box']
    ch, cw = crop.shape[:2]
    s = max((x2 - x1) / cw, (y2 - y1) / ch) * rng.uniform(1.12, 1.25)
    return bg, pegar(bg, crop, mask, (x1 + x2) / 2, (y1 + y2) / 2, cw * s, ch * s)


def procedural(huevos, rng):
    """1 o 2 huevos sobre un fondo procedural, con tamaño y posición al azar."""
    W, H = rng.choice([(640, 640), (640, 480), (480, 640), (288, 640)])
    bg = fondo_procedural(rng, W, H)
    cajas = []
    for _ in range(2 if rng.random() < 0.3 else 1):
        item = rng.choice(huevos)
        crop, mask = recortar_huevo(item)
        crop = variar_huevo(crop, rng)
        ch, cw = crop.shape[:2]
        s = rng.uniform(0.25, 0.7) * min(W, H) / max(ch, cw)
        w, h = cw * s, ch * s
        for _intento in range(10):
            cx, cy = rng.uniform(w / 2, W - w / 2), rng.uniform(h / 2, H - h / 2)
            if all(cx + w / 2 < a or cx - w / 2 > c or cy + h / 2 < b or cy - h / 2 > d for _, a, b, c, d in cajas):
                caja = pegar(bg, crop, mask, cx, cy, w, h)
                if caja:
                    cajas.append((item['cls'], *caja))
                break
    return bg, cajas


def generar_sinteticas(split_dir, out_dir, n_a, n_b, n_p, seed, prefijo):
    """A: Intact del montaje sobre fondos de otras fuentes. B: Crack de otras fuentes sobre el montaje.
    P: huevos de ambas clases sobre fondos procedurales. Solo usa huevos y fondos del mismo split."""
    rng = random.Random(seed)
    items = cargar_split(split_dir)
    intact = [i for i in items if i['cls'] == 1]
    crack_otras = [i for i in items if i['cls'] == 0 and i['fuente'] == 'otras']
    montaje = [i for i in items if i['fuente'] == 'montaje']
    (out_dir / 'images').mkdir(parents=True, exist_ok=True)
    (out_dir / 'labels').mkdir(parents=True, exist_ok=True)
    conteo = Counter()
    tareas = [('A', n_a, crack_otras, intact), ('B', n_b, montaje, crack_otras)]
    for tipo, n, fondos, huevos in tareas:
        hechos = 0
        orden = (huevos * (n // len(huevos) + 1))[:n] if tipo == 'A' else [rng.choice(huevos) for _ in range(n)]
        for k, huevo in enumerate(orden):
            for _intento in range(5):
                bg, caja = intercambio(rng.choice(fondos), huevo, rng)
                if caja:
                    break
            if not caja:
                continue
            bg, cajas = retocar(bg, [(huevo['cls'], *caja)], rng)
            guardar(out_dir, f'{prefijo}_{tipo}_{k:05d}', bg, cajas)
            conteo[(tipo, CLASSES[huevo['cls']])] += 1
            hechos += 1
    # mitad Intact / mitad Crack para que el fondo procedural no favorezca a ninguna clase
    pools = [intact, [i for i in items if i['cls'] == 0]]
    for k in range(n_p):
        bg, cajas = procedural(pools[k % 2], rng)
        if not cajas:
            continue
        bg, cajas = retocar(bg, cajas, rng)
        guardar(out_dir, f'{prefijo}_P_{k:05d}', bg, cajas)
        for c, *_ in cajas:
            conteo[('P', CLASSES[c])] += 1
    return conteo

## 3. Construir el dataset v2
En `/content/eggs_v2s` (se reconstruye igual cada vez, con semillas fijas):
- `train`: train original **sin** las copias `_dup` + sintéticas hechas solo con huevos y fondos de train.
- `valid`: valid original + sintéticas de valid (así `best.pt` se elige también por robustez al fondo).
- `test_synth`: sintéticas hechas solo con test, para medir el atajo. El test original no se toca.

In [ ]:
import pandas as pd

if V2_DIR.exists():
    shutil.rmtree(V2_DIR)

def copiar_originales(split, destino):
    for sub in ['images', 'labels']:
        (destino / sub).mkdir(parents=True, exist_ok=True)
    for img in (DATA_DIR / split / 'images').iterdir():
        if img.suffix.lower() in IMG_EXTS and '_dup' not in img.stem:
            shutil.copy2(img, destino / 'images' / img.name)
            shutil.copy2(DATA_DIR / split / 'labels' / f'{img.stem}.txt', destino / 'labels' / f'{img.stem}.txt')

conteos = {}
copiar_originales('train', V2_DIR / 'train')
conteos['train'] = generar_sinteticas(DATA_DIR / 'train', V2_DIR / 'train', 1228, 700, 700, seed=0, prefijo='syn_train')
copiar_originales('valid', V2_DIR / 'valid')
conteos['valid'] = generar_sinteticas(DATA_DIR / 'valid', V2_DIR / 'valid', 356, 200, 200, seed=1, prefijo='syn_valid')
conteos['test_synth'] = generar_sinteticas(DATA_DIR / 'test', V2_DIR / 'test_synth', 168, 168, 200, seed=2, prefijo='syn_test')

def escribir_yaml(nombre, **splits):
    p = V2_DIR / nombre
    with open(p, 'w') as f:
        yaml.safe_dump({'path': str(V2_DIR), **splits, 'nc': 2, 'names': CLASSES}, f, sort_keys=False)
    return p

DATA_V2 = escribir_yaml('data.yaml', train=str(V2_DIR / 'train/images'), val=str(V2_DIR / 'valid/images'),
                        test=str(DATA_DIR / 'test/images'))
DATA_TEST_ALL = escribir_yaml('test_all.yaml', train=str(V2_DIR / 'train/images'),
                              val=[str(DATA_DIR / 'test/images'), str(V2_DIR / 'test_synth/images')])

def resumen(split_dir):
    c = {}
    for lp in (split_dir / 'labels').glob('*.txt'):
        for line in lp.read_text().splitlines():
            if line.strip():
                k = CLASSES[int(line.split()[0])]
                c[k] = c.get(k, 0) + 1
    return c

filas = {s: resumen(V2_DIR / s) for s in ['train', 'valid', 'test_synth']}
display(pd.DataFrame(filas).T.assign(ratio=lambda d: (d['Crack'] / d['Intact']).round(2)))
print({s: dict(c) for s, c in conteos.items()})

## 4. Ejemplos de imágenes sintéticas
6 de cada tipo (A, B, P) de train con sus cajas: rojo = Crack, verde = Intact.

In [ ]:
import matplotlib.pyplot as plt

def dibujar_gt(img_path, labels_dir):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    for line in (labels_dir / f'{img_path.stem}.txt').read_text().splitlines():
        if line.strip():
            c, x, y, w, h = line.split()
            x, y, w, h = map(float, (x, y, w, h))
            color = (255, 0, 0) if c == '0' else (0, 200, 0)
            cv2.rectangle(img, (int((x - w / 2) * W), int((y - h / 2) * H)),
                          (int((x + w / 2) * W), int((y + h / 2) * H)), color, max(1, W // 200))
    return img

rnd = random.Random(0)
imgs = sorted((V2_DIR / 'train/images').glob('syn_train_*.jpg'))
muestra = [p for t in 'ABP' for p in rnd.sample([q for q in imgs if f'_{t}_' in q.name], 6)]
fig, axes = plt.subplots(3, 6, figsize=(20, 11))
for ax, p in zip(axes.flat, muestra):
    ax.imshow(dibujar_gt(p, V2_DIR / 'train/labels'))
    ax.set_title(p.stem, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Diagnóstico de v1: ¿usa el fondo para decidir?
Evalúa `v1` imagen por imagen (conf 0.5, IoU ≥ 0.5) separando por grupo. Si acierta en el test original pero falla en **A** (Intact fuera del montaje) y **B** (Crack dentro del montaje), el modelo está mirando el fondo y no el huevo.

In [ ]:
from ultralytics import YOLO

CONF = 0.5

def grupo_de(stem, clase_real):
    for t, nombre in [('_A_', 'A: Intact fuera del montaje'), ('_B_', 'B: Crack en el montaje'), ('_P_', 'P: fondo procedural')]:
        if stem.startswith('syn_') and t in stem:
            return nombre
    fuente = 'montaje' if stem.startswith('ec_egg') else 'otras fuentes'
    return f'orig: {clase_real} {fuente}'

def iou(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0

def leer_gt(labels_dir, stem):
    out = []
    for line in (labels_dir / f'{stem}.txt').read_text().splitlines():
        p = line.split()
        if len(p) == 5:
            c, xc, yc, w, h = int(p[0]), *map(float, p[1:])
            out.append((c, [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]))
    return out

def evaluar_imagenes(model, split_dir, conf=CONF):
    """Una fila por imagen: correcta si cada huevo tiene una predicción de su clase (IoU>=0.5) y no sobra nada."""
    filas = []
    # la carpeta (no una lista) para que prediga en lotes de 16 y no todas las imágenes a la vez en la GPU
    for r in model.predict(str(split_dir / 'images'), conf=conf, imgsz=640, batch=16, stream=True, verbose=False):
        path = Path(r.path)
        gt = leer_gt(split_dir / 'labels', path.stem)
        preds = list(zip(r.boxes.cls.int().tolist(), r.boxes.conf.tolist(), r.boxes.xyxyn.tolist()))
        errores, usadas = [], set()
        for c, gbox in gt:
            solapan = [j for j, (_, _, b) in enumerate(preds) if iou(gbox, b) >= 0.5]
            usadas.update(solapan)
            clases = {preds[j][0] for j in solapan}
            if not solapan:
                errores.append(f'{CLASSES[c]} no detectado')
            elif c not in clases:
                errores.append(f'{CLASSES[c]} -> {CLASSES[1 - c]}')
            elif len(clases) > 1:
                errores.append('doble clase')
        errores += [f'sobra {CLASSES[pc]}' for j, (pc, _, _) in enumerate(preds) if j not in usadas]
        clase_real = CLASSES[gt[0][0]] if gt else 'fondo'
        filas.append({'img': path, 'grupo': grupo_de(path.stem, clase_real), 'correcto': not errores,
                      'errores': errores, 'gt': gt, 'preds': preds})
    return pd.DataFrame(filas)

def tabla_grupos(df):
    return df.groupby('grupo')['correcto'].agg(acierto='mean', correctas='sum', imagenes='count').round(3)

model_v1 = YOLO(str(RUNS_DIR / BASE_RUN / 'weights/best.pt'))
df_v1 = pd.concat([evaluar_imagenes(model_v1, DATA_DIR / 'test'), evaluar_imagenes(model_v1, V2_DIR / 'test_synth')])
diag_v1 = tabla_grupos(df_v1)
display(diag_v1)

## 6. Entrenar v2 (parte de los pesos de v1)
Mismos aumentos que `v1`, con el dataset v2. Parte de `v1/best.pt` en vez de `yolov8n.pt`, así converge antes (60 épocas máx., `patience=15`). `cache='disk'` guarda las imágenes ya decodificadas en `/content` para acelerar cada época. Si `runs/v2` ya existe, se detiene en vez de sobrescribirlo.

In [ ]:
assert not (RUNS_DIR / NEW_RUN).exists(), f'{RUNS_DIR / NEW_RUN} ya existe: cambia NEW_RUN o usa la celda de reanudar.'

model = YOLO(str(RUNS_DIR / BASE_RUN / 'weights/best.pt'))
model.train(
    data=str(DATA_V2),
    imgsz=640, epochs=60, patience=15, batch=-1, cache='disk',
    project=str(RUNS_DIR), name=NEW_RUN,
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
    degrees=15, translate=0.15, scale=0.5,
    fliplr=0.5, flipud=0.2,
    mosaic=1.0, mixup=0.1,
)

## 6b. Reanudar v2 si se cortó la sesión
Si Colab se desconecta: ejecuta las celdas 1, 2 y 3 (reconstruyen el dataset idéntico) y luego esta. No ejecutes la 6.

In [ ]:
last = RUNS_DIR / NEW_RUN / 'weights/last.pt'
assert last.exists(), f'No encuentro {last}'
YOLO(str(last)).train(resume=True)

## 7. Comparar v1 y v2 y elegir modelo
- Métricas por clase en el **test original** (el mismo que se usó para `v1`).
- Acierto por grupo, incluyendo las sintéticas de test (A, B, P).

Se elige `v2` si mejora claramente en las sintéticas sin empeorar el test original (mAP50-95 no baja más de 0.02). El umbral se calcula con la curva F1 de Crack sobre test original + sintéticas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def metricas_clase(model, data, nombre):
    m = model.val(data=str(data), split='test' if data == DATA_V2 else 'val', imgsz=640, batch=16,
                  plots=True, project=str(RUNS_DIR), name=nombre, verbose=False)
    filas = {}
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        filas[CLASSES[c]] = {'P': p, 'R': r, 'mAP50': ap50, 'mAP50-95': ap}
    return m, pd.DataFrame(filas).T

model_v2 = YOLO(str(RUNS_DIR / NEW_RUN / 'weights/best.pt'))
comparacion = {}
for nombre, mdl in [(BASE_RUN, model_v1), (NEW_RUN, model_v2)]:
    _, t = metricas_clase(mdl, DATA_V2, f'{nombre}_test_orig')
    comparacion[nombre] = t
display(pd.concat(comparacion, axis=1).round(3))

df_v2 = pd.concat([evaluar_imagenes(model_v2, DATA_DIR / 'test'), evaluar_imagenes(model_v2, V2_DIR / 'test_synth')])
grupos = pd.concat({BASE_RUN: tabla_grupos(df_v1)['acierto'], NEW_RUN: tabla_grupos(df_v2)['acierto']}, axis=1)
display(grupos)

es_synth = grupos.index.str.match('[ABP]:')
mejora_synth = grupos.loc[es_synth, NEW_RUN].mean() - grupos.loc[es_synth, BASE_RUN].mean()
caida_orig = comparacion[BASE_RUN]['mAP50-95'].mean() - comparacion[NEW_RUN]['mAP50-95'].mean()
ELEGIDO = NEW_RUN if (mejora_synth > 0.05 and caida_orig < 0.02) else BASE_RUN
print(f'Sintéticas: {mejora_synth:+.3f} de acierto medio con v2 | test original: mAP50-95 {-caida_orig:+.3f}')
print(f'-> Modelo elegido: {ELEGIDO}')

model_final = model_v2 if ELEGIDO == NEW_RUN else model_v1
m_all, t_all = metricas_clase(model_final, DATA_TEST_ALL, f'{ELEGIDO}_test_all')
print('Test original + sintéticas:')
display(t_all.round(3))

px, k = m_all.box.px, {c: i for i, c in enumerate(m_all.box.ap_class_index)}
f1c = m_all.box.f1_curve[k[0]]
zona = np.where(f1c >= f1c.max() - 0.01)[0]
CONF_APP = round(float(px[zona[len(zona) // 2]]), 2)
print(f'Meseta F1 Crack: {px[zona[0]]:.2f}–{px[zona[-1]]:.2f} -> umbral recomendado CONF_APP = {CONF_APP}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(px, m_all.box.p_curve[k[0]], label='precisión Crack')
ax.plot(px, m_all.box.r_curve[k[0]], label='recall Crack')
ax.plot(px, f1c, lw=2, label='F1 Crack')
ax.plot(px, m_all.box.f1_curve[k[1]], ls=':', label='F1 Intact')
ax.axvline(CONF_APP, c='k', ls='--', label=f'recomendado {CONF_APP}')
ax.set_xlabel('umbral de confianza'); ax.set_ylim(0, 1.02); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.show()

## 8. Errores del modelo elegido
Hasta 12 imágenes mal clasificadas (test original + sintéticas). Caja blanca = etiqueta real, roja = predicción.

In [ ]:
import matplotlib.pyplot as plt

df_final = df_v2 if ELEGIDO == NEW_RUN else df_v1
errores = df_final[~df_final['correcto']]
print(f'{len(errores)} imágenes con error de {len(df_final)}')
print(errores['grupo'].value_counts())

def dibujar_pred(fila):
    img = cv2.cvtColor(cv2.imread(str(fila['img'])), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    t = max(1, round(min(H, W) / 150))
    for _, b in fila['gt']:
        cv2.rectangle(img, (int(b[0] * W), int(b[1] * H)), (int(b[2] * W), int(b[3] * H)), (255, 255, 255), t)
    for pc, pconf, b in fila['preds']:
        cv2.rectangle(img, (int(b[0] * W), int(b[1] * H)), (int(b[2] * W), int(b[3] * H)), (255, 0, 0), t)
        cv2.putText(img, f'{CLASSES[pc]} {pconf:.2f}', (int(b[0] * W), max(int(b[1] * H) - 4, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4 * t, (255, 0, 0), t)
    return img

if len(errores):
    muestra = errores.sample(min(12, len(errores)), random_state=0)
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for ax, (_, fila) in zip(axes.flat, muestra.iterrows()):
        ax.imshow(dibujar_pred(fila))
        ax.set_title(f"{fila['grupo']}\n{', '.join(fila['errores'])}", fontsize=8, color='red')
    for ax in axes.flat:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 9. Exportar a TFLite (LiteRT) con entrada NHWC
En ultralytics 8.4 el formato `tflite` se reemplazó por `litert`, que por defecto genera la entrada en NCHW `[1, 3, 640, 640]`. La cámara (vision-camera-resize-plugin) entrega NHWC `[1, 640, 640, 3]`, así que se añade una transposición dentro del modelo: la app no tiene que reordenar píxeles en JavaScript. Se exportan dos variantes:
- `eggs_<run>_fp32.tflite`: la más precisa; en el celular corre en GPU (Android) o Core ML (iOS) en FP16.
- `eggs_<run>_int8.tflite`: 4 veces más liviana, para CPU; calibrada con imágenes de valid.

Se guardan en `MyDrive/eggs_v2/exports/<run>` (si ya existe, se detiene).

In [ ]:
import torch
import ultralytics.utils.export.litert as litert_mod
from ultralytics.utils.export.engine import _NormalizeCoords

_torch2litert_original = litert_mod.torch2litert

class _EntradaNHWC(torch.nn.Module):
    """Recibe [1, H, W, 3] (lo que entrega la cámara) y lo pasa al modelo como [1, 3, H, W]."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x.permute(0, 3, 1, 2))

class _CalibracionNHWC:
    """Pasa las imágenes de calibración INT8 a NHWC para que coincidan con la nueva entrada."""
    def __init__(self, loader):
        self.loader = loader
    def __iter__(self):
        for batch in self.loader:
            batch = dict(batch)
            batch['img'] = batch['img'].permute(0, 2, 3, 1)
            yield batch

def _torch2litert_nhwc(model, im, file, quantize, calibration_dataset, metadata, prefix):
    h, w = int(im.shape[2]), int(im.shape[3])
    model = _EntradaNHWC(_NormalizeCoords(model, h, w, 'detect', len((metadata or {}).get('names', {})), None))
    normalize_original = litert_mod._NormalizeCoords
    litert_mod._NormalizeCoords = lambda m, *a, **k: m  # ya normalizado arriba
    try:
        return _torch2litert_original(
            model, im.permute(0, 2, 3, 1).contiguous(), file, quantize,
            _CalibracionNHWC(calibration_dataset) if calibration_dataset is not None else None, metadata, prefix)
    finally:
        litert_mod._NormalizeCoords = normalize_original

litert_mod.torch2litert = _torch2litert_nhwc

EXPORT_DIR = EXPORTS_DIR / ELEGIDO
assert not EXPORT_DIR.exists(), f'{EXPORT_DIR} ya existe: no se sobrescribe.'
work = Path('/content/export') / ELEGIDO
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)
pt = work / f'eggs_{ELEGIDO}.pt'
shutil.copy2(RUNS_DIR / ELEGIDO / 'weights/best.pt', pt)

f_fp32 = Path(YOLO(str(pt)).export(format='litert', imgsz=640))
f_int8 = Path(YOLO(str(pt)).export(format='litert', imgsz=640, quantize=8, data=str(DATA_V2), fraction=0.5))

EXPORT_DIR.mkdir(parents=True)
TFLITE = {'fp32': EXPORT_DIR / f'eggs_{ELEGIDO}_fp32.tflite', 'int8': EXPORT_DIR / f'eggs_{ELEGIDO}_int8.tflite'}
shutil.copy2(f_fp32, TFLITE['fp32'])
shutil.copy2(f_int8, TFLITE['int8'])
shutil.copy2(pt, EXPORT_DIR / pt.name)
for k, f in TFLITE.items():
    print(f'{k}: {f}  ({f.stat().st_size / 1e6:.1f} MB)')

## 10. Verificar los .tflite y guardar el resumen para la app
- Compara `.pt`, FP32 e INT8 en test original + sintéticas (deberían dar casi lo mismo).
- Imprime la entrada y salida reales del `.tflite` (forma, tipo) y comprueba a mano la decodificación que usará la app sobre una imagen.
- Guarda `resumen.json` junto a los `.tflite` con todo lo que necesita la documentación.

In [ ]:
import json
from ai_edge_litert.interpreter import Interpreter

verif = {}
for nombre, peso in [('pt', pt), ('fp32', TFLITE['fp32']), ('int8', TFLITE['int8'])]:
    m = YOLO(str(peso), task='detect').val(data=str(DATA_TEST_ALL), split='val', imgsz=640,
                                          batch=16 if nombre == 'pt' else 1, plots=False, verbose=False,
                                          project=str(RUNS_DIR), name=f'{ELEGIDO}_verif_{nombre}')
    verif[nombre] = {'mAP50': m.box.map50, 'mAP50-95': m.box.map,
                     **{f'mAP50-95 {CLASSES[c]}': m.box.class_result(i)[3] for i, c in enumerate(m.box.ap_class_index)},
                     'ms/img (Colab CPU)' if nombre != 'pt' else 'ms/img (GPU)': m.speed['inference']}
display(pd.DataFrame(verif).T.round(4))

it = Interpreter(model_path=str(TFLITE['fp32']))
it.allocate_tensors()
inp, out = it.get_input_details()[0], it.get_output_details()[0]
print('Entrada:', inp['shape'].tolist(), inp['dtype'].__name__, '| Salida:', out['shape'].tolist(), out['dtype'].__name__)

# Decodificación como en la app: imagen -> 640x640 RGB float 0..1 -> salida [1, 6, 8400]
ejemplo = sorted((DATA_DIR / 'test/images').iterdir())[0]
img = cv2.cvtColor(cv2.imread(str(ejemplo)), cv2.COLOR_BGR2RGB)
x = cv2.resize(img, (640, 640)).astype(np.float32)[None] / 255.0
it.set_tensor(inp['index'], x)
it.invoke()
y = it.get_tensor(out['index'])[0]            # (6, 8400)
scores = y[4:]                                # (2, 8400) ya en 0..1
i = int(scores.max(axis=0).argmax())
cx, cy, bw, bh = y[:4, i]
print(f'{ejemplo.name}: clase {CLASSES[int(scores[:, i].argmax())]} conf {scores[:, i].max():.3f} '
      f'caja cx={cx:.3f} cy={cy:.3f} w={bw:.3f} h={bh:.3f} (normalizada 0..1)')
print('Etiqueta real:', (DATA_DIR / 'test/labels' / f'{ejemplo.stem}.txt').read_text().strip())

resumen_app = {
    'modelo': ELEGIDO, 'clases': {0: 'Crack', 1: 'Intact'}, 'conf_recomendado': CONF_APP,
    'entrada': {'shape': inp['shape'].tolist(), 'dtype': inp['dtype'].__name__, 'layout': 'NHWC RGB 0..1'},
    'salida': {'shape': out['shape'].tolist(), 'dtype': out['dtype'].__name__,
               'filas': ['cx', 'cy', 'w', 'h', 'score_Crack', 'score_Intact']},
    'archivos_MB': {k: round(f.stat().st_size / 1e6, 2) for k, f in TFLITE.items()},
    'verificacion': {k: {m: round(float(v), 4) for m, v in d.items()} for k, d in verif.items()},
    'test_original_por_clase': comparacion.get(ELEGIDO, t_all).round(4).to_dict(),
    'acierto_por_grupo': grupos[ELEGIDO].round(4).to_dict(),
}
(EXPORT_DIR / 'resumen.json').write_text(json.dumps(resumen_app, indent=2, ensure_ascii=False))
print(json.dumps(resumen_app, indent=2, ensure_ascii=False))